In [2]:
import requests
import pandas as pd
from pytrends.request import TrendReq

print("--- 1. FETCHING HISTORICAL COVID-19 CLINICAL DATA ---")
# Querying the archived COVID19Bharat time-series repository
url = "https://data.incovid19.org/v4/min/timeseries.min.json"
response = requests.get(url)
data = response.json()

# Filter strictly for Delhi ('DL') during April 2020
delhi_dates = data['DL']['dates']
records = []
for date, metrics in delhi_dates.items():
    if date.startswith("2020-04"): 
        daily_cases = metrics.get('delta', {}).get('confirmed', 0)
        records.append({'date': date, 'daily_cases': daily_cases})

covid_df = pd.DataFrame(records)
covid_df['date'] = pd.to_datetime(covid_df['date']).dt.strftime('%Y-%m-%d')
print(f"Extracted {len(covid_df)} days of clinical data.")

print("\n--- 2. FETCHING HISTORICAL GOOGLE TRENDS DATA ---")
# Querying Google Trends for the exact same historical window
pytrends = TrendReq(hl='en-IN', tz=330)
pytrends.build_payload(["covid symptoms"], timeframe='2020-04-01 2020-04-30', geo='IN-DL')
trends_df = pytrends.interest_over_time().reset_index()

# Clean and rename search metrics
trends_df['date'] = pd.to_datetime(trends_df['date']).dt.strftime('%Y-%m-%d')
trends_df = trends_df.rename(columns={'covid symptoms': 'Search_Volume'})
if 'isPartial' in trends_df.columns:
    trends_df = trends_df.drop(columns=['isPartial'])

print("\n--- 3. ALIGNED HISTORICAL DATASET ---")
# Merge datasets on matching dates
master_df = pd.merge(covid_df, trends_df, on='date', how='inner')

# Display exactly 15 rows for inspection
display(master_df.head(15))

--- 1. FETCHING HISTORICAL COVID-19 CLINICAL DATA ---
Extracted 30 days of clinical data.

--- 2. FETCHING HISTORICAL GOOGLE TRENDS DATA ---

--- 3. ALIGNED HISTORICAL DATASET ---


,date,daily_cases,Search_Volume
0,2020-04-01,32,92
1,2020-04-02,141,77
2,2020-04-03,93,64
3,2020-04-04,59,57
4,2020-04-05,58,68
5,2020-04-06,22,59
6,2020-04-07,51,67
7,2020-04-08,93,71
8,2020-04-09,51,62
9,2020-04-10,183,83
